# SECOP deviations: full pipeline on Google Colab

Runs the reproducible pipeline of the `secop-dev` package:

1. Clone the GitHub repository and install the pinned environment (Python 3.11, same library versions as the thesis).
2. Run the tests, including the regression tests against the thesis files.
3. Rebuild the training table from the raw extracts.
4. Tune and evaluate the models (resumable: if the session expires, run the same cell again).
5. Compare with the thesis metrics, run the scenario ANOVA and compute SHAP values.

Data stay in Google Drive; results are written to `outputs/` inside the Drive project folder.

In [ ]:
# --- Parameters -----------------------------------------------------------------
REPO_URL = "https://github.com/hfelizzola/Prediccion-Desviaci-n-Proyectos-de-Infraestructura.git"
BRANCH = "main"
DRIVE_PROJECT = "/content/drive/MyDrive/Colab Notebooks/Prediccion de Adiciones Proyectos de Infraestructura"
RUN_ID = "v1.1-colab"  # keep the same id to resume an interrupted run
REPO_DIR = "/content/secop-dev"
VENV = "/content/venv"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1. Code and environment

If the repository is private, store a GitHub token with read access as the Colab secret `GITHUB_TOKEN` (key icon in the left panel).

In [ ]:
import os
import subprocess


def repo_url_with_token(url):
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        return url.replace("https://", f"https://{token}@")
    except Exception:
        return url


if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url_with_token(REPO_URL), REPO_DIR], check=True)
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
# Isolated Python 3.11 environment with the exact versions of uv.lock
!pip install -q uv
!cd {REPO_DIR} && UV_PROJECT_ENVIRONMENT={VENV} uv sync -q --frozen --python 3.11 --extra dev
!{VENV}/bin/secop-dev --help | head -20

In [ ]:
os.environ["SECOP_DATA_DIR"] = f"{DRIVE_PROJECT}/Data"
os.environ["SECOP_TERRIDATA_DIR"] = f"{DRIVE_PROJECT}/terridata"
os.environ["SECOP_OUTPUT_DIR"] = f"{DRIVE_PROJECT}/outputs"
CLI = f"{VENV}/bin/secop-dev"
RUN_DIR = f"{DRIVE_PROJECT}/outputs/runs/{RUN_ID}"

## 2. Tests

Unit tests plus regression tests: the pipeline must reproduce the thesis files exactly.

In [ ]:
!cd {REPO_DIR} && {VENV}/bin/python -m pytest -q -p no:cacheprovider

## 3. Training table from the raw extracts

In [ ]:
!{CLI} data

## 4. Model tuning and evaluation

4 models x 5 scenarios x 2 targets, 30 Optuna trials each. Finished combinations are skipped, so this cell can be re-run after a disconnection.

In [ ]:
!{CLI} train --run-id {RUN_ID}

## 5. Comparison with the thesis, ANOVA and SHAP

In [ ]:
!{CLI} compare-v1 --run-id {RUN_ID}

In [ ]:
!{CLI} doe --run-id {RUN_ID}

In [ ]:
!{CLI} shap --run-id {RUN_ID}

In [ ]:
import pandas as pd

pd.read_csv(f"{RUN_DIR}/comparison_v1.csv").round(2)